# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!git clone https://github.com/Hussainhhgh/flyrank-ml-internship.git 2>/dev/null

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method choice:** Logistic Regression and Random Forest, compared side by side. My lane is scoring pages by decay risk, and both models output a probability I can rank by — matching the "scoring" task type from ML-03. Logistic Regression gives interpretable coefficients; Random Forest tests whether a non-linear model meaningfully beats it without overcomplicating the lane.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')

# Target: is the page declining (matches ML-07's proxy, never used as a feature)
df['target'] = (df['trend_direction'] == 'down').astype(int)

# Features: knowable at decision time, same signals as ML-07 baseline + a few more
features = ['content_age_days', 'avg_position', 'ctr', 'impressions_90d',
            'engagement_rate', 'search_volume']
df_model = df.dropna(subset=features + ['target'])
print(f"Rows available for modeling: {len(df_model)}")
print(df_model[features + ['target']].describe())


Rows available for modeling: 27532
       content_age_days  avg_position           ctr  impressions_90d  \
count      27532.000000   27532.00000  27532.000000     27532.000000   
mean         254.297036      17.13120      0.320956      5621.613359   
std          134.003585      15.25836      2.013341     17492.259609   
min           90.000000       0.00000      0.000000         1.000000   
25%          132.000000       6.70000      0.000000       141.000000   
50%          229.000000      11.60000      0.080000       905.000000   
75%          347.000000      23.30000      0.290000      4114.000000   
max          564.000000     245.00000    100.000000    517715.000000   

       engagement_rate  search_volume        target  
count     27532.000000   27532.000000  27532.000000  
mean          2.586094     158.882391      0.563889  
std           8.049822    1518.270825      0.495910  
min           0.000000       0.000000      0.000000  
25%           0.000000       0.000000      0.0

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design:** Grouped by client_id, not by row — 21 clients in train, 10 in test, with zero overlap. If the same client appeared in both train and test, the model could learn client-specific quirks (a particular client's content style, industry, or reporting pattern) rather than a generalizable pattern — exactly the leakage risk flagged in notebook 02's overfitting lesson. This gives 23,372 training rows and 4,160 test rows, and any model evaluated here is being tested on genuinely unseen clients, not just unseen rows.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split

clients = df_model['client_id'].unique()
train_clients, test_clients = train_test_split(clients, test_size=0.3, random_state=42)

train_df = df_model[df_model['client_id'].isin(train_clients)]
test_df = df_model[df_model['client_id'].isin(test_clients)]

X_train, y_train = train_df[features], train_df['target']
X_test, y_test = test_df[features], test_df['target']

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")


Train: 23372 rows, 21 clients
Test: 4160 rows, 10 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Results:** Baseline rule (ML-07) scored Precision@50 = 0.48 on this test split. Logistic Regression improved on it to 0.54, and Random Forest reached 0.66 — an 18-point lift over the hand-written rule and a 12-point lift over the linear model. This suggests the relationship between my features (age, position, CTR, impressions, engagement, search volume) and decline risk isn't purely linear — Random Forest's ability to capture interactions between signals (e.g. "stale AND mid-position AND low CTR" combinations) is doing real work here, not just adding complexity for its own sake.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

# --- Baseline rule (from ML-07), reproduced on test set ---
test_df = test_df.copy()
test_df['baseline_score'] = (
    test_df['freshness_tier'].isin(['31-90', '91-180']).astype(int) * 2 +
    test_df['position_tier'].isin(['striking', 'page_1', 'page_3_5']).astype(int) * 2
)
baseline_p50 = precision_at_k(test_df['target'].reset_index(drop=True),
                                test_df['baseline_score'].reset_index(drop=True))

# --- Logistic Regression ---
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(y_test.reset_index(drop=True), pd.Series(logreg_scores))

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(y_test.reset_index(drop=True), pd.Series(rf_scores))

results = pd.DataFrame({
    'Method': ['Baseline rule (ML-07)', 'Logistic Regression', 'Random Forest'],
    'Precision@50': [baseline_p50, logreg_p50, rf_p50]
})
print(results)


                  Method  Precision@50
0  Baseline rule (ML-07)          0.48
1    Logistic Regression          0.54
2          Random Forest          0.66


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Feature importance:** content_age_days (0.33) and impressions_90d (0.29) dominate, together accounting for over 60% of the Random Forest's decisions. avg_position (0.18) matters next, while search_volume, ctr, and engagement_rate contribute comparatively little (each under 10%). This is a shift from my Week-4 baseline, which leaned entirely on freshness_tier and position_tier — the model is finding that raw content_age_days and impressions_90d carry more signal than the tiered/bucketed versions I hand-picked.

**Where the model disagrees with the baseline:** Complete disagreement on the top 50 — the two methods share zero pages in their top-50 picks. Looking at a sample of pages the Random Forest flagged that the baseline missed: 4 of 5 sampled pages were correctly declining (target=1), and they share a pattern — mid-range age (138-148 days, which falls in my baseline's "31-90/91-180" bucket boundary or just outside it) combined with near-zero CTR and zero search_volume. My hand-written rule's binary bucket cutoffs likely missed these because they sit right at tier boundaries, while the Random Forest can weigh the *continuous* value of content_age_days directly rather than snapping it into a bucket.

**What this suggests:** The baseline's tiering (turning continuous signals into buckets) throws away real information — pages just outside a tier boundary get treated identically to pages far from it. This is a meaningful, explainable weakness in the rule-based approach, not just "the model is a black box that happens to score higher."

**Caveat:** avg_position has a documented "0 = no data" convention in this dataset (not literal rank zero), so any page with avg_position=0 in the training data is being treated as a highly-ranked page by the raw numeric feature, which could be quietly biasing the model's position weighting. Worth flagging as a limitation, not fixed here.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

# Feature importance from Random Forest
importances = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("Random Forest feature importances:")
print(importances)

# Where does the model disagree with the baseline?
test_df['rf_score'] = rf_scores
test_df_sorted_rf = test_df.sort_values('rf_score', ascending=False)
test_df_sorted_baseline = test_df.sort_values('baseline_score', ascending=False)

rf_top50_ids = set(test_df_sorted_rf.head(50)['content_id'])
baseline_top50_ids = set(test_df_sorted_baseline.head(50)['content_id'])

only_rf = rf_top50_ids - baseline_top50_ids
only_baseline = baseline_top50_ids - rf_top50_ids
print(f"\nPages RF flagged that baseline missed: {len(only_rf)}")
print(f"Pages baseline flagged that RF missed: {len(only_baseline)}")

# Look at a few RF-only picks to understand what it's catching
rf_only_rows = test_df[test_df['content_id'].isin(list(only_rf)[:5])][
    ['content_id', 'content_age_days', 'avg_position', 'ctr', 'search_volume', 'target']
]
print("\nSample of pages RF flagged that baseline missed:")
print(rf_only_rows)


Random Forest feature importances:
            feature  importance
0  content_age_days    0.331537
3   impressions_90d    0.290518
1      avg_position    0.184158
5     search_volume    0.082172
2               ctr    0.080617
4   engagement_rate    0.030997

Pages RF flagged that baseline missed: 50
Pages baseline flagged that RF missed: 50

Sample of pages RF flagged that baseline missed:
                 content_id  content_age_days  avg_position   ctr  \
2610   content_a8cd736c9c7b               144          23.2  0.03   
13025  content_1520bae94c77               138           2.4  0.00   
14485  content_96ff7449bc47               148          11.5  0.00   
21704  content_c18c25a0088c               148          11.4  0.00   
26266  content_ad77c88d8536               148          12.1  0.00   

       search_volume  target  
2610             0.0       1  
13025            0.0       1  
14485            0.0       1  
21704            0.0       1  
26266            0.0       0  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.